# 🧠 BigMart Sales Prediction - Linear Regression Model
This notebook demonstrates a **professional machine learning pipeline** for predicting sales using the **BigMart Sales dataset**.

---
## 🪜 Steps Covered
1. Import libraries
2. Load and inspect data
3. Handle missing values
4. Feature engineering
5. Encode categorical variables
6. Scale numerical features
7. Split dataset
8. Train Linear Regression model
9. Evaluate model performance
10. Save trained model

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Load dataset
df = pd.read_csv("bigmartsales-cleaned.csv")
df.head()


### 2️⃣ Handle Missing Values & Feature Engineering

In [ ]:

# Fill missing numerical values with mean
df['Item_Weight'].fillna(df['Item_Weight'].mean(), inplace=True)

# Fill missing categorical values with mode
df['Outlet_Size'].fillna(df['Outlet_Size'].mode()[0], inplace=True)
df['Outlet_Location_Type'].fillna(df['Outlet_Location_Type'].mode()[0], inplace=True)

# Create new feature: Outlet_Age
df['Outlet_Age'] = 2020 - df['Outlet_Establishment_Year']

# Extract Item_Category from Item_Identifier
df['Item_Category'] = df['Item_Identifier'].apply(lambda x: x[:2])
df['Item_Category'] = df['Item_Category'].map({'FD': 'Food', 'DR': 'Drinks', 'NC': 'Non-Consumable'})

df.head()


### 3️⃣ Define Features and Target

In [ ]:

X = df.drop(columns=['Item_Outlet_Sales'])
y = df['Item_Outlet_Sales']

# Identify categorical and numerical columns
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

cat_cols, num_cols


### 4️⃣ Preprocessing Pipeline

In [ ]:

# Preprocessing for numerical and categorical columns
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)


### 5️⃣ Split Data and Train Model

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build pipeline with preprocessing and model
model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('regressor', LinearRegression())])

# Train model
model.fit(X_train, y_train)


### 6️⃣ Evaluate Model

In [ ]:

# Predict
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Evaluate
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

print("Training RMSE:", rmse_train)
print("Testing RMSE:", rmse_test)
print("Training R2:", r2_train)
print("Testing R2:", r2_test)


### 7️⃣ Save the Model

In [ ]:

joblib.dump(model, "linear_model_bigmartsales.pkl")
print("✅ Model saved as linear_model_bigmartsales.pkl")
